# Phase 22 / Notebook 02 -- Phase 2 alignment distillation

Loads `checkpoint_phase1.pt`, then distils high-quality conversation and REAL
Minecraft trajectory data into `Thought -> Action -> Observation -> Answer`
byte transcripts, and saves `checkpoint_phase2.pt` locally + on the Hub.

Sources: `HuggingFaceH4/no_robots` (CC BY-NC 4.0 -- non-commercial, attribute),
`TucanoBR/Tucano-SFT` (PT-BR), `m-a-p/CodeFeedback-Filtered-Instruction`,
`TESS-Computer/minecraft-vla-stage3` and `vanyacohen/MET-Bench-Minecraft`.
There is **no verified HF dataset of human-authored textual Minecraft ReAct
trajectories**; the converter preserves real source actions/states and emits an
explicit `[source: no textual observation]` marker rather than inventing one.

Prompt/header bytes are masked from the loss (only the assistant / ReAct
continuation is trained), and the exact UTF-8 packing carry is checkpointed.

In [ ]:
import json
import os
import pathlib
import subprocess
import sys

REPO = pathlib.Path(os.environ.get("AIRESEARCH_REPO", "/content/airesearch"))
if not (REPO / ".git").exists():
    subprocess.check_call(["git", "clone", "https://github.com/gabrielbosse1/airesearch.git", str(REPO)])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "datasets>=3,<5", "huggingface_hub>=0.25,<2", "pyyaml", "tqdm"])
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("repo", REPO)

In [ ]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    print("GPU", torch.cuda.get_device_name(DEVICE),
          "capability", torch.cuda.get_device_capability(DEVICE),
          "bf16_supported", torch.cuda.is_bf16_supported())
else:
    print("WARNING: no CUDA device. This notebook must run on a Colab T4 GPU for real training.")

## Load the Phase 1 weights (strict phase order)

In [ ]:
from huggingface_hub import login

# Set these in Colab secrets / environment. The checkpoint repo is REQUIRED for
# robust resume when the Colab VM is recycled.
HF_REPO = os.environ.get("HF_REPO", "")
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(token=HF_TOKEN)
cfg["checkpoints"]["hf_repo"] = HF_REPO or None
cfg["checkpoints"]["resume_from_hf"] = bool(HF_REPO)
print("HF checkpoint repo", cfg["checkpoints"]["hf_repo"] or "(not set - local resume only)")

In [ ]:
from training.phase22_common import hf_download, load_config, load_checkpoint

cfg = load_config(CONFIG_PATH)
cfg["checkpoints"]["hf_repo"] = HF_REPO or None
ck = cfg["checkpoints"]
phase1_path = pathlib.Path(ck["local_dir"]) / ck["phase1_name"]
print("fetch phase 1", hf_download(ck.get("hf_repo"),
                                   f"{ck['hf_path_in_repo']}/{ck['phase1_name']}",
                                   phase1_path))
assert phase1_path.is_file(), "Run notebook 01 first, or set HF_REPO to resume from the Hub."

## Verify the alignment sources

In [ ]:
from data.phase22_sources import (SourceRegistry, minecraft_phase_coverage,
                                  stream_phase2)

registry = SourceRegistry()
transcripts = []
for transcript, prefix in stream_phase2(cfg, limit_rows_per_source=8, registry=registry):
    transcripts.append(transcript)
print("phase2 samples", len(transcripts))
print("ReAct labels", all(b"Thought:" in t and b"Action:" in t and
                          b"Observation:" in t and b"Answer:" in t
                          for t in transcripts if b"Provenance:" in t and b"Ladder:" in t))
print("phase ladder coverage", minecraft_phase_coverage(transcripts))
registry.manifest(cfg["data"]["license_manifest"],
                  extra={"phase": 2, "note": "alignment sources + ReAct conversion"})

## Phase 2 training

In [ ]:
from training.phase22_train import Phase22Trainer

MAX_STEPS = int(os.environ.get("PHASE2_MAX_STEPS", "0")) or None
trainer = Phase22Trainer(cfg, phase=2, device=DEVICE, data_mode="hub",
                         max_steps=MAX_STEPS, init_from_phase=1,
                         strict_phase_order=True)
report = trainer.run()
print(json.dumps({"steps": report["steps_completed"],
                  "initialisation": report["initialisation"],
                  "resumed": report["resumed"],
                  "checkpoint": report["checkpoint"]["path"],
                  "last_loss": report["last_loss"]}, indent=2))

In [ ]:
path = trainer._checkpoint_path()
assert path.is_file(), f"missing {path}"
print("checkpoint_phase2.pt bytes", path.stat().st_size)
ck = cfg["checkpoints"]
if ck.get("hf_repo"):
    from training.phase22_common import hf_upload
    print("upload", hf_upload(path, ck["hf_repo"],
                              f"{ck['hf_path_in_repo']}/checkpoint_phase2.pt"))
print("To RESUME: re-run notebook 02. Phase 1 weights are loaded first, then any")
print("checkpoint_phase2.pt state overrides them.")

## Resume

Re-running this notebook loads `checkpoint_phase1.pt` first (strict phase order),
then restores `checkpoint_phase2.pt` if it exists and skips completed steps.
If the VM is recycled, set `HF_REPO` and the previous checkpoints are fetched
from the Hub automatically.